In [1]:
# List data
!ls -R

.:
data_25.09.23  data_26.03.28  database_pipeline.ipynb
data_25.12.27  data_26.06.26  data_complete_zipped

./data_25.09.23:
calendar.csv  listings.csv.gz	  neighbourhoods.geojson  reviews.csv.gz
listings.csv  neighbourhoods.csv  reviews.csv

./data_25.12.27:
calendar.csv  listings.csv.gz	  neighbourhoods.geojson  reviews.csv.gz
listings.csv  neighbourhoods.csv  reviews.csv

./data_26.03.28:
calendar.csv  listings.csv.gz	  neighbourhoods.geojson  reviews.csv.gz
listings.csv  neighbourhoods.csv  reviews.csv

./data_26.06.26:
calendar.csv  listings.csv.gz	  neighbourhoods.geojson  reviews.csv.gz
listings.csv  neighbourhoods.csv  reviews.csv

./data_complete_zipped:
data_25.09.23.zip  data_25.12.27.zip  data_26.03.28.zip  data_26.06.26.zip


In [ ]:
# Imports
import pandas as pd
import psycopg2

In [ ]:
# Connection to database
conn = psycopg2.connect(
    host="localhost",
    database="airbnb_berlin_db",
    user="",
    password=""
)

In [4]:
table_names = ["listings","neighbourhoods","reviews","calendar"]
folder_names = ["data_25.09.23","data_25.12.27","data_26.03.28","data_26.06.26"]
folder_tables_columns ={folder_name: {table: "" for table in table_names} for folder_name in folder_names}
data_types = []

In [5]:
for folder in folder_names:
    print(f"################################\nFolder {folder} .csv-files:\n################################\n")
    for index, table in enumerate(table_names):
        print(f"{index+1}. table: {table}\n")
        file_path = folder + "/" + table + ".csv"
        data = pd.read_csv(file_path)
        folder_tables_columns[folder][table] = data.columns.tolist()
        print(data.head(5))
        print(data.dtypes)
        data_types.append([data[column].dtype for column in data.columns.tolist()])

################################
Folder data_25.09.23 .csv-files:
################################

1. table: listings

      id                         listing_url       scrape_id last_scraped  \
0   3176   https://www.airbnb.com/rooms/3176  20250923202926   2025-09-24   
1   9991   https://www.airbnb.com/rooms/9991  20250923202926   2025-09-24   
2  14325  https://www.airbnb.com/rooms/14325  20250923202926   2025-09-24   
3  17904  https://www.airbnb.com/rooms/17904  20250923202926   2025-09-24   
4  20858  https://www.airbnb.com/rooms/20858  20250923202926   2025-09-24   

        source                                           name  \
0  city scrape                Fabulous Flat in great Location   
1  city scrape            Geourgeous flat - outstanding views   
2  city scrape            Studio Apartment in Prenzlauer Berg   
3  city scrape  Beautiful Kreuzberg studio - 3 months minimum   
4  city scrape                  Designer Loft in Berlin Mitte   

                          

In [6]:
folder_tables_columns

{'data_25.09.23': {'listings': ['id',
   'listing_url',
   'scrape_id',
   'last_scraped',
   'source',
   'name',
   'description',
   'neighborhood_overview',
   'picture_url',
   'host_id',
   'host_url',
   'host_name',
   'host_since',
   'host_location',
   'host_about',
   'host_response_time',
   'host_response_rate',
   'host_acceptance_rate',
   'host_is_superhost',
   'host_thumbnail_url',
   'host_picture_url',
   'host_neighbourhood',
   'host_listings_count',
   'host_total_listings_count',
   'host_verifications',
   'host_has_profile_pic',
   'host_identity_verified',
   'neighbourhood',
   'neighbourhood_cleansed',
   'neighbourhood_group_cleansed',
   'latitude',
   'longitude',
   'property_type',
   'room_type',
   'accommodates',
   'bathrooms',
   'bathrooms_text',
   'bedrooms',
   'beds',
   'amenities',
   'price',
   'minimum_nights',
   'maximum_nights',
   'minimum_minimum_nights',
   'maximum_minimum_nights',
   'minimum_maximum_nights',
   'maximum_maximum

In [ ]:
# Sanity check
for table in table_names:
    print(f"\n'{table}'-columns in all folders the same?:\n")
    print(
        folder_tables_columns[folder_names[0]][table] ==
        folder_tables_columns[folder_names[1]][table] ==
        folder_tables_columns[folder_names[2]][table] ==
        folder_tables_columns[folder_names[3]][table]
    )



'listings'-columns in all folders the same?:

False

'neighbourhoods'-columns in all folders the same?:

True

'reviews'-columns in all folders the same?:

True

'calendar'-columns in all folders the same?:

False


In [9]:
# unique data types
data_types_flat = [data_type for sub_list in data_types for data_type in sub_list]
data_types_set = set(data_types_flat) 
print(data_types_set)

{dtype('int64'), <StringDtype(storage='python', na_value=nan)>, dtype('float64')}


In [ ]:
# read official metadata for column type mapping:
# https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit?usp=sharing&pli=1&authuser=0
metadata = pd.read_excel(
    "Inside_Airbnb_Data_Dictionary.xlsx",
    sheet_name=["listings.csv detail v4.3", "reviews.csv v1", "calendar.csv v2"],
    header=7,
    usecols=["Field", "Type"]
)

In [40]:
print(metadata["listings.csv detail v4.3"])

                Field      Type
0                  id   integer
1         listing_url      text
2           scrape_id    bigint
3        last_scraped  datetime
4              source      text
..                ...       ...
74  reviews_per_month   numeric
75                NaN       NaN
76     Change control       NaN
77              Field    Change
78             source     Added

[79 rows x 2 columns]


In [47]:
# remove footer data of variable length
for sheet in metadata:
    footer_start_idx = metadata[sheet].index[metadata[sheet]['Field']=='Change control'][0]-1
    metadata[sheet] = metadata[sheet].iloc[:footer_start_idx]

In [59]:
# unique datatypes
metadata_types = [
    metadata[sheet]["Type"].unique().tolist() for sheet in metadata
]
unique_metadata_types = set([metadata_type for metadata_table in metadata_types for metadata_type in metadata_table])
print(unique_metadata_types)

{'integer', 'numeric', 'json', 'boolean', 'text', 'datetime', 'currency', 'boolean [t=true; f=false]', 'date', nan, 'bigint', 'string'}


In [ ]:
# data type mapping for values without 1:1-equivalent in postgres according to docs: 
# https://www.postgresql.org/docs/current/datatype.html
dtype_mapping = {
    "nan": "text",
    "datetime": "timestamp",
    "currency": "text",
    "boolean [t=true; f=false]": "boolean",
    "string": "text"
}

In [ ]:
# Create schemas for database 
query = '''
    CREATE SCHEMA IF NOT EXISTS data_25_09_23 AUTHORIZATION airbnb_berlin;
    CREATE SCHEMA IF NOT EXISTS data_25_12_27 AUTHORIZATION airbnb_berlin;
    CREATE SCHEMA IF NOT EXISTS data_26_03_28 AUTHORIZATION airbnb_berlin;
    CREATE SCHEMA IF NOT EXISTS data_26_06_26 AUTHORIZATION airbnb_berlin;
 '''

In [ ]:
with conn:
    with conn.cursor() as cursor:
        cursor.execute(query)

In [ ]:
# Create tables from actual data
for folder in folder_names:
    schema_name = folder.replace(".","_")
    for table in table_names:
        create_table_query = f"CREATE TABLE IF NOT EXISTS {schema_name}.{table} ("
        if table in [element.split(".", 1)[0] for element in list(metadata.keys())]:
            metadata_table = metadata[[element for element in metadata if element.startswith(table)][0]]
            for column_name in folder_tables_columns[folder][table]:
                if column_name in list(metadata_table["Field"]):
                    column_type = str(list(metadata_table["Type"].loc[metadata_table["Field"]==column_name])[0])
                    if column_type in list(dtype_mapping.keys()):
                        create_table_query += f'"{column_name}" {dtype_mapping[column_type]},'
                    else:
                        if column_name in ['id', 'host_id']: #prevent overflow from big numbers
                            create_table_query += f'"{column_name}" bigint,'
                        else:
                            create_table_query += f'"{column_name}" {column_type},'
                else:
                    create_table_query += f'"{column_name}" TEXT,'
        else:
            for element in folder_tables_columns[folder][table]:
                create_table_query += f"{element} TEXT,"
        create_table_query=create_table_query[:-1]+");"
        print(create_table_query)
        try:
            with conn:
                with conn.cursor() as cursor:
                    cursor.execute(create_table_query)
        except psycopg2.Error as e:
            print(f"Fehler beim Erstellen: {e}")


CREATE TABLE IF NOT EXISTS data_25_09_23.listings ("id" integer,"listing_url" text,"scrape_id" bigint,"last_scraped" timestamp,"source" text,"name" text,"description" text,"neighborhood_overview" text,"picture_url" text,"host_id" integer,"host_url" text,"host_name" text,"host_since" date,"host_location" text,"host_about" text,"host_response_time" text,"host_response_rate" text,"host_acceptance_rate" text,"host_is_superhost" boolean,"host_thumbnail_url" text,"host_picture_url" text,"host_neighbourhood" text,"host_listings_count" text,"host_total_listings_count" text,"host_verifications" text,"host_has_profile_pic" boolean,"host_identity_verified" boolean,"neighbourhood" text,"neighbourhood_cleansed" text,"neighbourhood_group_cleansed" text,"latitude" numeric,"longitude" numeric,"property_type" text,"room_type" text,"accommodates" integer,"bathrooms" numeric,"bathrooms_text" text,"bedrooms" integer,"beds" integer,"amenities" json,"price" text,"minimum_nights" integer,"maximum_nights" int

In [190]:
# Populate the table
for folder in folder_names:
    schema_name = folder.replace(".", "_")
    for table in table_names:
        file_path = f"{folder}/{table}.csv"
        copy_query = f"COPY {schema_name}.{table} FROM STDIN WITH (FORMAT csv, HEADER true, NULL '')"
        try:
            with open(file_path, "r") as file:
                with conn:
                    with conn.cursor() as cursor:
                        cursor.execute(f"TRUNCATE TABLE {schema_name}.{table};")
                        cursor.copy_expert(sql=copy_query, file=file)
            print(f"Fertig: {schema_name}.{table}")
        except psycopg2.Error as e:
            print(f"Fehler beim Import von {schema_name}.{table}: {e}")

Fertig: data_25_09_23.listings
Fertig: data_25_09_23.neighbourhoods
Fertig: data_25_09_23.reviews
Fertig: data_25_09_23.calendar
Fertig: data_25_12_27.listings
Fertig: data_25_12_27.neighbourhoods
Fertig: data_25_12_27.reviews
Fertig: data_25_12_27.calendar
Fertig: data_26_03_28.listings
Fertig: data_26_03_28.neighbourhoods
Fertig: data_26_03_28.reviews
Fertig: data_26_03_28.calendar
Fertig: data_26_06_26.listings
Fertig: data_26_06_26.neighbourhoods
Fertig: data_26_06_26.reviews
Fertig: data_26_06_26.calendar
